# CD Training Dump Viewer

Visualises the images dumped by `./build/cd_training_test`.

For each batch index *b* in both `dump_cd/lns_vert/` and `dump_cd/lns_horiz/`:
- **E** — centre-cropped input image `E_crop[b]`
- **shrink_ref** — `E_crop[b] + coeff_c · R_ref[b]`  (reference FFT path)
- **shrink_test** — `E_crop[b] + coeff_c · R_test[b]` (accelerated path under test)
- **diff** — pointwise difference `shrink_test − shrink_ref`

Run the binary first:
```bash
./build/cd_training_test --sigma 20.0 --N 256 --B 30 --coeff_c 0.1 --dump_max 4
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pathlib, re

DUMP_ROOT = pathlib.Path('../dump_cd')
assert DUMP_ROOT.exists(), f'Run ./build/cd_training_test first — {DUMP_ROOT} not found'

def load_image(path):
    """Load a col-major float32 dump (shape H W); returns (H, W) numpy array."""
    path = pathlib.Path(path)
    meta = {}
    for line in path.read_text().splitlines():
        if not line.startswith('#'): break
        m = re.match(r'# shape:\s*(\d+)\s+(\d+)', line)
        if m: meta['H'], meta['W'] = int(m.group(1)), int(m.group(2))
    arr = np.loadtxt(path, comments='#')  # (H, W) where rows=cols, cols=rows (col-major)
    return arr.T   # transpose so arr[row, col] for standard image display

def available_batches(section_dir):
    d = DUMP_ROOT / section_dir
    idxs = sorted({int(re.search(r'b(\d+)_', f.name).group(1))
                   for f in d.glob('b*_E.txt')})
    return idxs

print('Sections found:', [p.name for p in DUMP_ROOT.iterdir() if p.is_dir()])

In [ ]:
def plot_section(section, batches=None):
    d = DUMP_ROOT / section
    all_b = available_batches(section)
    if batches is None:
        batches = all_b

    for b in batches:
        E          = load_image(d / f'b{b}_E.txt')
        shrink_ref  = load_image(d / f'b{b}_shrink_ref.txt')
        shrink_test = load_image(d / f'b{b}_shrink_test.txt')
        diff        = shrink_test - shrink_ref

        vmax_e = np.abs(E).max()
        vmax_s = max(np.abs(shrink_ref).max(), np.abs(shrink_test).max())
        dmax   = np.abs(diff).max()

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        kw_e = dict(cmap='RdBu_r', vmin=-vmax_e, vmax=vmax_e, interpolation='nearest')
        kw_s = dict(cmap='RdBu_r', vmin=-vmax_s, vmax=vmax_s, interpolation='nearest')
        kw_d = dict(cmap='bwr',    vmin=-dmax,   vmax=dmax,   interpolation='nearest')

        im0 = axes[0].imshow(E,          **kw_e); axes[0].set_title(f'E_crop  (b={b})')
        im1 = axes[1].imshow(shrink_ref,  **kw_s); axes[1].set_title('shrink_ref  (E + c·R_ref)')
        im2 = axes[2].imshow(shrink_test, **kw_s); axes[2].set_title('shrink_test (E + c·R_test)')
        im3 = axes[3].imshow(diff,        **kw_d); axes[3].set_title(f'diff (max={dmax:.2e})')

        for ax, im in zip(axes, [im0, im1, im2, im3]):
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        fig.suptitle(f'{section}  b={b}  W={E.shape[1]}', fontsize=13)
        plt.tight_layout()
        out = DUMP_ROOT / section / f'b{b}_compare.png'
        plt.savefig(out, dpi=120)
        plt.show()
        print(f'  saved {out}')

plot_section('lns_vert')

In [ ]:
plot_section('lns_horiz')

## Center-row / center-col profile comparison

Extract the center row (lns_vert) or center column (lns_horiz) from each image to compare the 1-D signal shape.

In [ ]:
def plot_profile(section, center_row=True, batches=None):
    d = DUMP_ROOT / section
    all_b = available_batches(section)
    if batches is None:
        batches = all_b

    n = len(batches)
    fig, axes = plt.subplots(n, 1, figsize=(10, 3*n), squeeze=False)

    for i, b in enumerate(batches):
        E           = load_image(d / f'b{b}_E.txt')
        shrink_ref  = load_image(d / f'b{b}_shrink_ref.txt')
        shrink_test = load_image(d / f'b{b}_shrink_test.txt')
        W = E.shape[1]
        mid = W // 2

        if center_row:
            e_line   = E[mid, :]
            ref_line = shrink_ref[mid, :]
            tst_line = shrink_test[mid, :]
            xlabel = 'col'
        else:
            e_line   = E[:, mid]
            ref_line = shrink_ref[:, mid]
            tst_line = shrink_test[:, mid]
            xlabel = 'row'

        ax = axes[i, 0]
        ax.plot(e_line,   label='E_crop',      lw=1.5, ls=':')
        ax.plot(ref_line, label='shrink_ref',  lw=2)
        ax.plot(tst_line, label='shrink_test', lw=1.5, ls='--')
        ax.axhline(0, color='k', lw=0.5)
        ax.set_xlabel(xlabel); ax.set_ylabel('value')
        ax.set_title(f'{section}  b={b}  center-{"row" if center_row else "col"} profile')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out = DUMP_ROOT / section / 'profiles.png'
    plt.savefig(out, dpi=120)
    plt.show()
    print(f'saved {out}')

plot_profile('lns_vert',  center_row=True)
plot_profile('lns_horiz', center_row=False)